In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import Statevector


# =========================================================
# Helper functions
# =========================================================

def bitstr_le(x, k):
    return [(x >> b) & 1 for b in range(k)]

def valid_window_indicator(n_bits, M, a):
    return 1 if all(b == 0 for b in n_bits[a:a+M]) else 0

def compute_g_list(n_bits, M):
    W = len(n_bits) - M + 1
    return [valid_window_indicator(n_bits, M, a) for a in range(W)]

def prepare_uniform_anchors(qc, a_reg, num_anchor_states):
    k = len(a_reg)
    amps = np.zeros(2**k, dtype=complex)
    amps[:num_anchor_states] = 1 / np.sqrt(num_anchor_states)
    qc.append(StatePreparation(amps, label="PrepA"), a_reg)

def apply_anchor_mask(qc, a_reg, a_value):
    """Turn condition anchor==a_value into all-1 controls."""
    k = len(a_reg)
    bits = bitstr_le(a_value, k)
    for b in range(k):
        if bits[b] == 0:
            qc.x(a_reg[b])

def undo_anchor_mask(qc, a_reg, a_value):
    apply_anchor_mask(qc, a_reg, a_value)

def compute_branch_value_into_g(qc, a_reg, p_reg, g_reg, g_values, branch_value):
    """
    branch_value=0 -> compute g(a)
    branch_value=1 -> compute g(a+1)
    """
    p = p_reg[0]
    g = g_reg[0]
    num_anchor_states = len(g_values) - 1

    for a in range(num_anchor_states):
        target = g_values[a] if branch_value == 0 else g_values[a+1]
        if target == 0:
            continue

        apply_anchor_mask(qc, a_reg, a)

        # control on p=branch_value
        if branch_value == 0:
            qc.x(p)

        controls = list(a_reg) + [p]
        qc.mcx(controls, g)

        if branch_value == 0:
            qc.x(p)

        undo_anchor_mask(qc, a_reg, a)

def pretty_print_state(sv, N, k, tol=1e-10):
    """
    Circuit order assumed: n, a, p, g
    Qiskit printed bit order: g p a_{k-1}...a_0 n_{N-1}...n_0
    """
    total = int(np.log2(len(sv.data)))
    print("State grouped as |a>|p>|g> (n omitted in the label):\n")

    for idx, amp in enumerate(sv.data):
        if abs(amp) <= tol:
            continue

        bits = format(idx, f"0{total}b")
        g_bit = bits[0]
        p_bit = bits[1]
        a_bits = bits[2:2+k]
        n_bits = bits[2+k:][::-1]

        a_int = int(a_bits, 2) if k > 0 else 0
        print(f"amp={amp: .6g}   |a={a_bits} ({a_int})>|p={p_bit}>|g={g_bit}>   n={n_bits}")

def print_boundary_branch(sv, N, k, num_anchor_states, tol=1e-10):
    """
    After the final H on g, the useful boundary subspace is:
        p=0, g=1

    because:
        |g(a)> - |g(a+1)>
    becomes (after H on g):
        0          if g(a)=g(a+1)
        ±sqrt(2)|1> if g(a)!=g(a+1)
    """
    total = int(np.log2(len(sv.data)))

    print("\nUseful boundary branch: p=0, g=1\n")
    survivors = []

    for idx, amp in enumerate(sv.data):
        if abs(amp) <= tol:
            continue

        bits = format(idx, f"0{total}b")
        g_bit = bits[0]
        p_bit = bits[1]
        a_bits = bits[2:2+k]
        a_int = int(a_bits, 2) if k > 0 else 0

        if p_bit == '0' and g_bit == '1' and a_int < num_anchor_states:
            survivors.append((a_int, a_bits, amp))

    if not survivors:
        print("No surviving anchors.")
        return

    survivors.sort(key=lambda x: x[0])
    for a_int, a_bits, amp in survivors:
        print(f"a={a_int:2d}   bits={a_bits}   amp={amp: .6g}")

    print("\nThese are exactly the anchors where g(a) != g(a+1).")


# =========================================================
# Build corrected circuit
# =========================================================

def build_boundary_difference_circuit_1d(n_bits, M):
    N = len(n_bits)
    W = N - M + 1
    if W < 2:
        raise ValueError("Need at least two windows.")

    g_values = compute_g_list(n_bits, M)
    num_anchor_states = W - 1
    k = int(np.ceil(np.log2(num_anchor_states))) if num_anchor_states > 1 else 1

    n_reg = QuantumRegister(N, "n")
    a_reg = QuantumRegister(k, "a")
    p_reg = QuantumRegister(1, "p")
    g_reg = QuantumRegister(1, "g")

    qc = QuantumCircuit(n_reg, a_reg, p_reg, g_reg)

    # -----------------------------------------------------
    # Encode classical occupancy
    # -----------------------------------------------------
    for i, b in enumerate(n_bits):
        if b == 1:
            qc.x(n_reg[i])

    # -----------------------------------------------------
    # Uniform anchors a = 0,...,W-2
    # -----------------------------------------------------
    prepare_uniform_anchors(qc, a_reg, num_anchor_states)

    # -----------------------------------------------------
    # Path ancilla in |-> = (|0>-|1>)/sqrt(2)
    # -----------------------------------------------------
    qc.x(p_reg[0])
    qc.h(p_reg[0])

    qc.barrier()

    # -----------------------------------------------------
    # Compute g(a) on branch p=0, g(a+1) on branch p=1
    # -----------------------------------------------------
    compute_branch_value_into_g(qc, a_reg, p_reg, g_reg, g_values, branch_value=0)
    compute_branch_value_into_g(qc, a_reg, p_reg, g_reg, g_values, branch_value=1)

    qc.barrier()

    # -----------------------------------------------------
    # Recombine path
    #
    # p=0 branch now contains the difference:
    #   |g(a)> - |g(a+1)>
    # -----------------------------------------------------
    qc.h(p_reg[0])

    # -----------------------------------------------------
    # NEW crucial step:
    # Hadamard on g compresses the difference vector into g=1
    #
    # If g(a)=g(a+1): cancellation -> zero in p=0,g=1
    # If g(a)!=g(a+1): survives in p=0,g=1
    # -----------------------------------------------------
    qc.h(g_reg[0])

    return qc, g_values, W, k


# =========================================================
# Worked example
# =========================================================

n_bits = [1, 1, 0, 0, 0, 0, 0, 1, 1]
M = 3

qc, g_values, W, k = build_boundary_difference_circuit_1d(n_bits, M)

print("Occupancy array n:")
print(n_bits)
print(f"\nWindow size M = {M}")
print(f"Number of windows W = {W}")
print("g(a)=1 iff window a is all zeros:\n")

for a in range(W):
    print(f"a={a}: window={n_bits[a:a+M]} -> g(a)={g_values[a]}")

print("\nExpected boundary anchors are those with g(a) != g(a+1):")
for a in range(W-1):
    print(f"a={a}: g(a)={g_values[a]}, g(a+1)={g_values[a+1]}, boundary={g_values[a] != g_values[a+1]}")

print("\nCircuit:")
print(qc.draw("text"))

sv = Statevector.from_instruction(qc)

print("\n================ FULL VIEW ================\n")
pretty_print_state(sv, N=len(n_bits), k=k)

print("\n================ BOUNDARY BRANCH ================\n")
print_boundary_branch(sv, N=len(n_bits), k=k, num_anchor_states=W-1)

In [ ]:
sv = Statevector.from_instruction(qc)
num_anchor_states = W - 1

analyze_difference_branch_grouped(
    sv,
    num_n_qubits=len(n_bits),
    k=k,
    num_anchor_states=num_anchor_states
)

In [ ]:
qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")